# 🧱 Build `mini.py` — a transformer from scratch

One file, grown one piece at a time, from a single multiply into a tiny but **real** transformer that plays fill-in-the-blank. **Pure Python — no numpy, no PyTorch.** Run the cells top to bottom; each one adds a rung.

By the end it will learn `dog↔bark`, `cat↔meow`, `fish↔swim` from masked examples alone — and you'll have written every line.

## Rung 1 — a model is just numbers
The simplest model: one weight. "Learning" is nudging it until the guess matches reality.

In [1]:
w = 0.2
def model(x): return w * x          # guess a tip from a bill

print("start:  w =", w, " guess for a £50 bill =", model(50))

bill, real_tip = 50, 8.0            # the real tip was £8, not £10
for step in range(2001):
    err = model(bill) - real_tip    # how wrong is the guess?
    w  -= 0.0001 * err * bill       # nudge w to shrink the error
print("learned: w =", round(w, 3), " guess for £50 =", round(model(50), 2))

start:  w = 0.2  guess for a £50 bill = 10.0
learned: w = 0.16  guess for £50 = 8.0


## Rung 2 — the autograd engine
A `Value` is a number that **remembers how it was built**, so it can hand back its gradient — "which way to nudge me." This ~30-line class is all of autograd, and it powers everything below.

In [2]:
import random, math

class Value:
    def __init__(self, data, _children=()):
        self.data = data; self.grad = 0.0
        self._backward = lambda: None; self._prev = set(_children)
    def __add__(self, o):
        o = o if isinstance(o, Value) else Value(o); out = Value(self.data + o.data, (self, o))
        def _b(): self.grad += out.grad; o.grad += out.grad
        out._backward = _b; return out
    def __mul__(self, o):
        o = o if isinstance(o, Value) else Value(o); out = Value(self.data * o.data, (self, o))
        def _b(): self.grad += o.data * out.grad; o.grad += self.data * out.grad
        out._backward = _b; return out
    def __pow__(self, k):
        out = Value(self.data ** k, (self,))
        def _b(): self.grad += k * (self.data ** (k - 1)) * out.grad
        out._backward = _b; return out
    def relu(self):
        out = Value(self.data if self.data > 0 else 0.0, (self,))
        def _b(): self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _b; return out
    def exp(self):
        out = Value(math.exp(self.data), (self,))
        def _b(): self.grad += out.data * out.grad
        out._backward = _b; return out
    def log(self):
        out = Value(math.log(self.data), (self,))
        def _b(): self.grad += (1.0 / self.data) * out.grad
        out._backward = _b; return out
    def __neg__(self): return self * -1
    def __sub__(self, o): return self + (o * -1 if isinstance(o, Value) else Value(-o))
    def __radd__(self, o): return self + o
    def __rmul__(self, o): return self * o
    def backward(self):
        topo, seen = [], set()
        def build(v):
            if v not in seen:
                seen.add(v)
                for c in v._prev: build(c)
                topo.append(v)
        build(self); self.grad = 1.0
        for v in reversed(topo): v._backward()

**Does it really reproduce Rung 1's nudge?** Re-do that one step with `Value`s and ask the engine for `w.grad` — it should match the by-hand answer.

In [3]:
w = Value(0.2)
pred = w * 50                 # guess for a £50 bill
err  = pred + (-8.0)          # pred - real_tip
loss = (err * err) * 0.5      # squared error
loss.backward()               # the machine fills in every .grad
print("w.grad      =", w.grad)
print("by hand (L1):", (pred.data - 8.0) * 50)

w.grad      = 100.0
by hand (L1): 100.0


## Rungs 3–4 — vectors, layers, and the bend (ReLU)
Real inputs are **vectors**; a layer is a **dot product per neuron**; `ReLU` adds the bend that lets a network leave straight lines behind. That `Linear → ReLU` block is the feed-forward part of a transformer.

In [4]:
def dot(a, b):                       # a . b for two vectors of Values
    s = a[0] * b[0]
    for i in range(1, len(a)): s = s + a[i] * b[i]
    return s
def linear(x, W):    return [dot(row, x) for row in W]            # no bias
def linear_b(x, W, b): return [dot(W[i], x) + b[i] for i in range(len(W))]

x = [Value(1.0), Value(-2.0)]
W = [[Value(1.0), Value(1.0)], [Value(2.0), Value(-1.0)]]
h = [n.relu() for n in linear_b(x, W, [Value(0.0), Value(0.0)])]
print("layer output after ReLU:", [round(v.data, 2) for v in h])

layer output after ReLU: [0.0, 4.0]


## Softmax & cross-entropy
Two more pieces, built from the same engine: `softmax` turns scores into a 100% budget; `cross_entropy` measures how surprised the model is at the right answer (log-sum-exp form, so it never takes `log(0)`).

In [5]:
def softmax(zs):
    m = max(z.data for z in zs)
    es = [(z - m).exp() for z in zs]
    s = es[0]
    for e in es[1:]: s = s + e
    return [e * (s ** -1) for e in es]

def cross_entropy(logits, target):
    m = max(z.data for z in logits)
    es = [(z - m).exp() for z in logits]
    S = es[0]
    for e in es[1:]: S = S + e
    return (S.log() + m) - logits[target]   # = -log P(target)

## Rung 5 — words become vectors (the corpus)
One small vocabulary, three hidden rules. The model must rediscover them from blanks.

In [6]:
VOCAB = ["<pad>", "<mask>", "dog", "cat", "fish", "bark", "meow", "swim"]
TOK = {w: i for i, w in enumerate(VOCAB)}
V = len(VOCAB)
PAIRS = [("dog", "bark"), ("cat", "meow"), ("fish", "swim")]
print(VOCAB)

['<pad>', '<mask>', 'dog', 'cat', 'fish', 'bark', 'meow', 'swim']


## Rungs 7–9 — the model: embeddings + positions + Q/K/V attention + head
Everything clicks together here. Each token becomes an embedding plus a position; attention lets the masked slot look around (query·key, ÷√d, softmax, mix the values); a final head turns the blended vector into one score per vocab word.

In [7]:
D = 8
SCALE = 1.0 / math.sqrt(D)

def _init(seed=1):
    random.seed(seed)
    rv = lambda: Value(random.uniform(-0.3, 0.3))
    mat = lambda r, c: [[rv() for _ in range(c)] for _ in range(r)]
    return {"emb": mat(V, D), "pos": mat(2, D),
            "Wq": mat(D, D), "Wk": mat(D, D), "Wv": mat(D, D),
            "Wh": mat(V, D), "bh": [Value(0.0) for _ in range(V)]}

def params(P):
    out = []
    for k in ("emb", "pos", "Wq", "Wk", "Wv", "Wh"):
        for row in P[k]: out += row
    return out + P["bh"]

def forward(P, tokens, mask_pos):
    X = [[P["emb"][TOK[t]][k] + P["pos"][i][k] for k in range(D)] for i, t in enumerate(tokens)]
    Q = [linear(x, P["Wq"]) for x in X]
    K = [linear(x, P["Wk"]) for x in X]
    Vv = [linear(x, P["Wv"]) for x in X]
    i = mask_pos
    w = softmax([dot(Q[i], K[j]) * SCALE for j in range(len(X))])
    ctx = [sum((w[j] * Vv[j][k] for j in range(1, len(X))), w[0] * Vv[0][k]) for k in range(D)]
    return linear_b(ctx, P["Wh"], P["bh"])

def examples():
    data = []
    for a, b in PAIRS:
        data.append((["<mask>", b], 0, TOK[a]))   # ___ bark -> dog
        data.append(([a, "<mask>"], 1, TOK[b]))   # dog  ___ -> bark
    return data

def predict(P, tokens, mask_pos):
    p = softmax(forward(P, tokens, mask_pos))
    return VOCAB[max(range(V), key=lambda i: p[i].data)]

## Before training — it knows nothing
Random weights → random guesses.

In [8]:
P = _init()
print("BEFORE:", [f"{a} ___ -> {predict(P, [a, chr(60)+chr(109)+chr(97)+chr(115)+chr(107)+chr(62)], 1)}" for a, _ in PAIRS])

BEFORE: ['dog ___ -> dog', 'cat ___ -> cat', 'fish ___ -> <mask>']


## Train it — watch the loss crash
The same 5-line loop from Lesson 1: guess → measure loss → `backward()` → nudge every weight → repeat. (~a few seconds.)

In [9]:
ps = params(P)
data = examples()
for step in range(400):
    total = 0.0
    for toks, mpos, target in data:
        loss = cross_entropy(forward(P, toks, mpos), target); total += loss.data
        for p in ps: p.grad = 0.0
        loss.backward()
        for p in ps: p.data -= 0.1 * p.grad
    if step % 100 == 0 or step == 399:
        print(f"step {step:4d}   loss {total/len(data):.3f}")

step    0   loss 2.124


step  100   loss 0.005


step  200   loss 0.001


step  300   loss 0.001


step  399   loss 0.000


## After training — it rediscovered all three rules
Nobody told it `dog` goes with `bark`. It found that from blanks alone.

In [10]:
print("AFTER (animal -> sound):", [f"{a} ___ -> {predict(P, [a, '<mask>'], 1)}" for a, _ in PAIRS])
print("AFTER (sound -> animal):", [f"___ {b} -> {predict(P, ['<mask>', b], 0)}" for _, b in PAIRS])
print("parameters:", len(params(P)))

AFTER (animal -> sound): ['dog ___ -> bark', 'cat ___ -> meow', 'fish ___ -> swim']
AFTER (sound -> animal): ['___ bark -> dog', '___ meow -> cat', '___ swim -> fish']
parameters: 344


## Rung 10 — the one change that makes it *write* (the causal mask)
For generating text GPT-style, a word may only look **left** (never at the future). That single rule is the causal mask.

In [11]:
words = ["the", "dog", "says", "___"]
print("each word can only attend to itself and the words before it:")
for i, wd in enumerate(words):
    seen = softmax([Value(1.0 if j <= i else -1e9) for j in range(4)])
    print(f"  {wd:4s} -> {[round(x.data, 2) for x in seen]}")

each word can only attend to itself and the words before it:
  the  -> [1.0, 0.0, 0.0, 0.0]
  dog  -> [0.5, 0.5, 0.0, 0.0]
  says -> [0.33, 0.33, 0.33, 0.0]
  ___  -> [0.25, 0.25, 0.25, 0.25]


## What you built

From `tip = 0.2 * bill` to a working transformer — embeddings, positions, Q/K/V attention, a head, trained by masked language modelling on a from-scratch autograd engine. **Every line yours.**

This is the same recipe as BERT and GPT — just tiny. Make `D` bigger, stack the block many times, train on the whole internet, and you have ChatGPT. Same file, more of it.